In [13]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tqdm import tqdm
from sklearn.metrics import confusion_matrix

In [21]:
def filter_df(df, label_map, N):
    nums_df = [(df.query(f'label == {num}').iloc[:N] / 255)
               .assign(label=[label] * N) 
               for label, num in label_map.items()]
    return pd.concat(nums_df).reset_index(drop=True)

raw_train = (pd.read_csv('data/mnist_train.csv', header=None)
         .rename(columns={0: 'label'}))
raw_test = (pd.read_csv('data/mnist_test.csv', header=None)
        .rename(columns={0: 'label'}))

label_map = {1: 1, -1: 7}
train_df = filter_df(raw_train, label_map, 2000)
test_df = filter_df(raw_test, label_map, 500)

In [ ]:
def plot(losses, constraints, title=None):
    fig = make_subplots(rows=1, cols=2)

    fig.add_trace(
        go.Scatter(x=np.arange(len(losses)), y=losses, mode="lines", name="Loss"),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=np.arange(len(constraints)),
            y=constraints,
            mode="lines",
            name="Constraint Violation",
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        showlegend=True,
        height=400,
        width=1000,
        title_text=f"Loss and Constraint Violation vs. Iteration {title and f'({title})'}",
        title_x=0.5,
        xaxis_title_text="Iteration",
        xaxis2_title_text="Iteration",
        yaxis_title_text="Loss",
        yaxis2_title_text="Constraint Violation",
    )

    fig.update_yaxes(type="log", row=1, col=1)
    fig.update_yaxes(type="log", row=1, col=2)

    fig.show()

In [ ]:
def display_confusion_matrix(fig, w, df, df_type, row, col):
    X = df.iloc[:, 1:].to_numpy()
    Y = df.iloc[:, 0].to_numpy()
    predictions = np.sign(np.dot(X, w))

    cm = confusion_matrix(Y, predictions, labels=[1, -1])
    cm_percentages = cm / cm.sum(axis=1, keepdims=True) * 100
    accuracy = (Y == predictions).mean() * 100

    heatmap = go.Heatmap(
        z=cm,
        x=[1, -1],
        y=[1, -1],
        colorscale="Blues",
        text=[
            [f"{cm[i, j]}<br>({cm_percentages[i, j]:.1f}%)" for j in range(2)]
            for i in range(2)
        ],
        texttemplate="%{text}",
        showscale=False,
    )

    fig.add_trace(heatmap, row=row, col=col)
    fig.update_xaxes(
        title_text="Predicted Label",
        tickmode="array",
        tickvals=[1, -1],
        row=row,
        col=col,
    )
    fig.update_yaxes(
        title_text="True Label", tickmode="array", tickvals=[1, -1], row=row, col=col
    )
    fig.update_yaxes(scaleanchor=f"x{col}", scaleratio=1, row=row, col=col)
    fig.layout.annotations[(row - 1) * 2 + (col - 1)].text = (
        f"{df_type} Confusion Matrix - Accuracy: {accuracy:.1f}%"
    )


def display_cm(w, test, train):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Test Confusion Matrix", "Train Confusion Matrix"],
    )

    display_confusion_matrix(fig, w, test, "Test", 1, 1)
    display_confusion_matrix(fig, w, train, "Train", 1, 2)

    fig.update_layout(height=500, width=900)
    fig.show()

In [ ]:
X = train_df.iloc[:, 1:].to_numpy()
Y = train_df.iloc[:, 0].to_numpy()
w0 = np.zeros(X.shape[1])
z0 = np.zeros(X.shape[1])
u0 = np.zeros(X.shape[1])
Lambda = 3e-2
rho = 1e-1

F = lambda w, z: np.mean([np.log1p(np.exp(-y * np.dot(x, w)))
    for x, y in zip(X, Y)]) + (Lambda / 2) * np.dot(z, z)
grad_fw = lambda w, x, y: (-y * x) / (1 + np.exp(y * np.dot(x, w)))
grad_Fw = lambda w, z, u: np.mean([grad_fw(w, x, y) for x, y in zip(X, Y)], 
                                  axis=0) + rho * (w - z + u)

def grad_descent(w, z, u, step_size=5e-1, T=100, eps=1e-2):
    losses = np.array([F(w, z)])
    for _ in np.arange(T):
        w = w - step_size * grad_Fw(w, z, u)
        losses = np.append(losses, F(w, z))
        if np.linalg.norm(grad_Fw(w, z, u)) < eps:
            break
    # plot(losses, constraints=[])
    return w

def train(w, z, u, T=1000):
    losses = np.array([F(w, z)])
    constraints = np.array([np.linalg.norm(w - z)])
    for _ in tqdm(np.arange(T)):
        w = grad_descent(w, z, u)
        z = rho / (rho + Lambda) * (w + u)
        u = u + w - z
        losses = np.append(losses, F(w, z))
        constraints = np.append(constraints, np.linalg.norm(w - z))
    return w, losses, constraints

In [33]:
w, losses, constraints = train(w0, z0, u0, T=10)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:11<00:00,  1.12s/it]


In [34]:
plot(losses, constraints, title=f'Lambda = {Lambda}, rho = {rho}')

In [ ]:
display_cm(w, test_df, train_df)

: 